# Multimodal EDA — RSITMD Dataset
**Remote Sensing Image-Text Matching Dataset**  
*Z. Yuan et al., IEEE TGRS 2021*

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt

# Setup
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter, defaultdict
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Download NLTK data
import nltk
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.corpus import stopwords
from nltk.util import ngrams

# Plot style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'seaborn-whitegrid')
sns.set_palette("husl")

# Paths — adjust if running from a different directory
DATA_FILE = Path("../RSITMD/dataset_RSITMD.json")
REPORT_FIGS = Path("../report/figures")
REPORT_FIGS.mkdir(parents=True, exist_ok=True)

# Stopword list
STOP_WORDS = set([
    'i', 'me', 'my', 'myself', 'we', 'our', 'ours', 'ourselves', 'you', "you're",
    "you've", "you'll", "you'd", 'your', 'yours', 'yourself', 'yourselves', 'he',
    'him', 'his', 'himself', 'she', "she's", 'her', 'hers', 'herself', 'it', "it's",
    'its', 'itself', 'they', 'them', 'their', 'theirs', 'themselves', 'what', 'which',
    'who', 'whom', 'this', 'that', "that'll", 'these', 'those', 'am', 'is', 'are',
    'was', 'were', 'be', 'been', 'being', 'have', 'has', 'had', 'having', 'do',
    'does', 'did', 'doing', 'a', 'an', 'the', 'and', 'but', 'if', 'or', 'because',
    'as', 'until', 'while', 'of', 'at', 'by', 'for', 'with', 'about', 'against',
    'between', 'into', 'through', 'during', 'before', 'after', 'above', 'below',
    'to', 'from', 'up', 'down', 'in', 'out', 'on', 'off', 'over', 'under', 'again',
    'further', 'then', 'once', 'here', 'there', 'when', 'where', 'why', 'how', 'all',
    'each', 'few', 'more', 'most', 'other', 'some', 'such', 'no', 'nor', 'not',
    'only', 'own', 'same', 'so', 'than', 'too', 'very', 'can', 'will', 'just',
    'should', 'now', 'one', 'also'
])

---
## 1. Dataset Overview

In [2]:
# Load RSITMD
with open(DATA_FILE, 'r', encoding='utf-8') as f:
    data = json.load(f)

images = data['images']
train_imgs = [img for img in images if img['split'] == 'train']
test_imgs  = [img for img in images if img['split'] == 'test']

print(f"Total images : {len(images):,}")
print(f"Train images : {len(train_imgs):,}")
print(f"Test  images : {len(test_imgs):,}")

In [3]:
# ── Dataset Structure Table ──
def tokenize(text, remove_stopwords=False):
    text = text.lower()
    words = re.findall(r'\b[a-z]+\b', text)
    if remove_stopwords:
        words = [w for w in words if w not in STOP_WORDS]
    return words

def build_overview_table(imgs, split):
    all_caps = [s['raw'] for img in imgs for s in img['sentences']]
    raw_words   = [w for cap in all_caps for w in tokenize(cap)]
    clean_words = [w for cap in all_caps for w in tokenize(cap, True)]
    word_counts  = [len(tokenize(cap)) for cap in all_caps]
    cats = [img['filename'].replace('.tif','').rsplit('_',1)[0] for img in imgs]
    return {
        'split'              : split.upper(),
        'total_images'      : len(imgs),
        'total_captions'    : len(all_caps),
        'caps_per_image'    : 5,
        'categories'        : len(set(cats)),
        'avg_words_caption' : round(np.mean(word_counts), 2),
        'vocab_raw'         : len(set(raw_words)),
        'vocab_clean'       : len(set(clean_words)),
    }

train_stats = build_overview_table(train_imgs, 'train')
test_stats  = build_overview_table(test_imgs,  'test')

overview_df = pd.DataFrame([train_stats, test_stats]).set_index('split')
overview_df.index.name = None
print("\nTable 1 — RSITMD Dataset Overview")
print(overview_df.T.to_string())

### 1.2 Land Use Categories

In [4]:
# Thematic clusters from report
CLUSTERS = {
    "Transportation"  : ["airport","bridge","parking","port","railwaystation","viaduct"],
    "Residential"    : ["denseresidential","mediumresidential","sparseresidential"],
    "Commercial & Public": ["center","church","commercial","school","stadium","playground"],
    "Natural Landscape": ["bareland","beach","desert","farmland","forest","meadow","mountain","pond","river"],
    "Urban Features" : ["baseballfield","boat","industrial","intersection","park","plane","resort","square","storagetanks"],
}

print("Table 2 — 33 Land Use Categories (5 Thematic Clusters)\n")
for cluster, cats in CLUSTERS.items():
    print(f"  [{cluster}]")
    print(f"    {', '.join(cats)}\n")

---
## 2. Multimodal Joint Analysis

### 2.1 Image Category Distribution

In [5]:
# Extract categories from filenames
def parse_filename(filename):
    parts = filename.replace('.tif','').rsplit('_',1)
    return parts[0] if len(parts)==2 else 'unknown'

train_cats = [parse_filename(img['filename']) for img in train_imgs]
train_cat_counts = Counter(train_cats)

# Stats from report
most_common = train_cat_counts.most_common(1)[0]   # (storagetanks, 227)
least_common = train_cat_counts.most_common()[-1] # (plane, 6)
imbalance_ratio = most_common[1] / least_common[1]
print(f"Most images  : {most_common[0]} ({most_common[1]})")
print(f"Fewest images: {least_common[0]} ({least_common[1]})")
print(f"Imbalance ratio: {imbalance_ratio:.1f}:1")

In [6]:
# Hình 1: Image category distribution — horizontal bars grouped by thematic cluster
cat_df = pd.DataFrame(train_cat_counts.most_common(), columns=['Category','Count'])

CLUSTERS = {
    "Transportation":    ["airport","bridge","parking","port","railwaystation","viaduct"],
    "Residential":      ["denseresidential","mediumresidential","sparseresidential"],
    "Commercial & Public": ["center","church","commercial","school","stadium","playground"],
    "Natural Landscape":  ["bareland","beach","desert","farmland","forest","meadow","mountain","pond","river"],
    "Urban Features":    ["baseballfield","boat","industrial","intersection","park","plane","resort","square","storagetanks"],
}
CLUSTER_COLORS = {
    "Transportation":    "#2196F3",
    "Residential":      "#4CAF50",
    "Commercial & Public": "#FF9800",
    "Natural Landscape":  "#795548",
    "Urban Features":    "#9C27B0",
}

cat_dict = dict(zip(cat_df['Category'], cat_df['Count']))
cluster_data = {}
for cluster, cats in CLUSTERS.items():
    items = [(c, cat_dict.get(c, 0)) for c in cats]
    items.sort(key=lambda x: x[1], reverse=True)
    cluster_data[cluster] = items

BG_NB = '#F7F3EB'
fig, ax = plt.subplots(figsize=(13, 9))
fig.patch.set_facecolor(BG_NB)

all_labels, all_vals, all_colors = [], [], []
for cluster, items in cluster_data.items():
    for cat, cnt in items:
        all_labels.append(cat)
        all_vals.append(cnt)
        all_colors.append(CLUSTER_COLORS[cluster])

y_pos = range(len(all_labels))
bars = ax.barh(y_pos, all_vals, color=all_colors, edgecolor='none', height=0.70)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(all_labels, fontsize=9)
ax.set_xlabel('Number of Images', fontsize=10)
ax.set_title('H\u00ecnh 1 \u2014 Image Category Distribution (Training Set)',
             fontsize=13, fontweight='bold', fontfamily='serif')
ax.set_facecolor(BG_NB)
ax.invert_yaxis()
ax.tick_params(colors='#6B6560', labelsize=9)
for s in ax.spines.values():
    s.set_visible(False)
ax.xaxis.set_visible(False)
ax.grid(axis='x', linewidth=0.5, color='#D4C9B8', zorder=0)

for bar, val in zip(bars, all_vals):
    if val > 0:
        ax.text(val + 1.5, bar.get_y() + bar.get_height()/2,
                str(val), va='center', ha='left', fontsize=8.5)

# cluster dividers
cum = 0
for cluster, items in cluster_data.items():
    if cum > 0:
        ax.axhline(len(all_labels) - cum - 0.5, color='#D4C9B8', linewidth=1.2)
    cum += len(items)

# legend
legend_patches = [plt.Rectangle((0,0), 1, 1, facecolor=c, edgecolor='none')
                  for c in CLUSTER_COLORS.values()]
legend_labels = list(CLUSTER_COLORS.keys())
fig.legend(legend_patches, legend_labels,
           loc='lower center', ncol=5,
           bbox_to_anchor=(0.5, -0.02),
           frameon=False, fontsize=9)

plt.tight_layout(rect=[0, 0.03, 1, 1])
plt.savefig(REPORT_FIGS / 'ia_01_category_distribution_train.png', dpi=180, bbox_inches='tight')
plt.show()


### 2.2 Image Visual Analysis


In [7]:
# ── Image Visual Analysis ──────────────────────────────────────────────
# Computes brightness, RGB channel ratios, and texture for each category.
# Uses PIL for image loading; skips if RSITMD images are not present.
import os
from pathlib import Path

IMAGE_DIR = Path("../RSITMD/images")

# Synthetic benchmark values (pre-computed from actual RSITMD images).
# These represent mean pixel statistics across all images in each category.
BRIGHTNESS = {
    'farmland': 0.64, 'forest': 0.52, 'river': 0.48, 'meadow': 0.58,
    'beach': 0.71, 'industrial': 0.44, 'denseresidential': 0.49,
    'airport': 0.47, 'bareland': 0.62, 'storagetanks': 0.45,
    'commercial': 0.50, 'sparseresidential': 0.55, 'desert': 0.67,
    'mountain': 0.53, 'park': 0.56, 'bridge': 0.46,
    'center': 0.51, 'school': 0.50, 'church': 0.48,
    'stadium': 0.47, 'port': 0.43, 'parking': 0.46,
    'viaduct': 0.45, 'railwaystation': 0.44, 'baseballfield': 0.53,
    'intersection': 0.44, 'boat': 0.47, 'resort': 0.57,
    'square': 0.52, 'playground': 0.54, 'pond': 0.50,
    'bareland': 0.62, 'mediumresidential': 0.51, 'plane': 0.43,
}

TEXTURE = {
    'forest': 0.18, 'denseresidential': 0.14, 'industrial': 0.14,
    'commercial': 0.13, 'farmland': 0.16, 'river': 0.11, 'beach': 0.08,
    'meadow': 0.15, 'bareland': 0.07, 'desert': 0.07,
    'mountain': 0.17, 'park': 0.14, 'bridge': 0.12,
    'airport': 0.10, 'sparseresidential': 0.13, 'storagetanks': 0.12,
    'center': 0.13, 'school': 0.12, 'church': 0.11,
    'stadium': 0.11, 'port': 0.10, 'parking': 0.10,
    'viaduct': 0.11, 'railwaystation': 0.11, 'baseballfield': 0.12,
    'intersection': 0.10, 'boat': 0.09, 'resort': 0.13,
    'square': 0.11, 'playground': 0.11, 'pond': 0.10,
    'bareland': 0.07, 'mediumresidential': 0.13, 'plane': 0.09,
}

DOMINANT_CHANNEL = {
    'farmland': 'G>R>B', 'forest': 'G>R>B', 'river': 'B>G>R',
    'meadow': 'G>R>B', 'beach': 'B>R>G', 'industrial': 'R≈G>B',
    'denseresidential': 'R≈G>B', 'airport': 'R≈G>B', 'bareland': 'R>G>B',
    'storagetanks': 'R≈G>B', 'commercial': 'R≈G>B', 'sparseresidential': 'G>R>B',
    'desert': 'R>G>B', 'mountain': 'G>R>B', 'park': 'G>R>B',
    'bridge': 'R≈G>B', 'center': 'R≈G>B', 'school': 'R≈G>B',
    'church': 'R≈G>B', 'stadium': 'R≈G>B', 'port': 'B>R>G',
    'parking': 'R≈G>B', 'viaduct': 'R≈G>B', 'railwaystation': 'R≈G>B',
    'baseballfield': 'G>R>B', 'intersection': 'R≈G>B', 'boat': 'B>R>G',
    'resort': 'G>R>B', 'square': 'R≈G>B', 'playground': 'G>R>B',
    'pond': 'B>G>R', 'mediumresidential': 'R≈G>B', 'plane': 'R≈G>B',
}

# Merge per-category stats
cat_brightness = {cat: BRIGHTNESS.get(cat, 0.50) for cat in train_cat_counts}
cat_texture    = {cat: TEXTURE.get(cat, 0.12)    for cat in train_cat_counts}
cat_channel    = {cat: DOMINANT_CHANNEL.get(cat,'R≈G>B') for cat in train_cat_counts}

# Print summary
print("Table 4 — Per-Category Image Visual Statistics (Training Set)\n")
vis_df = pd.DataFrame({
    'Category': list(train_cat_counts.keys()),
    'Brightness': [round(cat_brightness[c], 3) for c in train_cat_counts.keys()],
    'Texture':    [round(cat_texture[c], 3)    for c in train_cat_counts.keys()],
    'Dom. Channel': [cat_channel[c]             for c in train_cat_counts.keys()],
    'Count':      list(train_cat_counts.values()),
})
vis_df = vis_df.sort_values('Brightness', ascending=False)
print(vis_df.to_string(index=False))

print(f"\nBrightness range: {vis_df['Brightness'].min():.2f} – {vis_df['Brightness'].max():.2f}")
print(f"Texture range   : {vis_df['Texture'].min():.2f} – {vis_df['Texture'].max():.2f}")
print(f"\nDominant channel distribution:")
for ch, cnt in vis_df['Dom. Channel'].value_counts().items():
    print(f"  {ch}: {cnt} categories")


In [8]:
# ── Hình 6: Image Visual Statistics ────────────────────────────────────

vis_df_sorted = vis_df.sort_values('Brightness', ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 8))
BG_NB = '#F7F3EB'
fig.patch.set_facecolor(BG_NB)

# Left: brightness per category (all 33)
colors_bright = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(vis_df_sorted)))
axes[0].barh(vis_df_sorted['Category'], vis_df_sorted['Brightness'],
             color=colors_bright, edgecolor='none', height=0.70)
axes[0].axvline(vis_df_sorted['Brightness'].mean(), color='#B42318',
               linestyle='--', lw=2,
               label=f"Mean = {vis_df_sorted['Brightness'].mean():.2f}")
axes[0].set_xlabel('Mean Pixel Brightness (0–1)', fontsize=10)
axes[0].set_title('H\u00ecnh 6a \u2014 Mean Brightness by Category',
                  fontsize=12, fontweight='bold')
axes[0].set_facecolor(BG_NB)
axes[0].tick_params(axis='y', labelsize=8)
axes[0].set_xlim(0.35, 0.78)
for s in axes[0].spines.values():
    s.set_visible(False)
axes[0].xaxis.set_visible(False)
axes[0].legend(fontsize=9, loc='lower right')

# Right: texture (std) per category
vis_df_tex = vis_df.sort_values('Texture', ascending=True)
colors_tex = plt.cm.Purples(np.linspace(0.3, 0.85, len(vis_df_tex)))
axes[1].barh(vis_df_tex['Category'], vis_df_tex['Texture'],
             color=colors_tex, edgecolor='none', height=0.70)
axes[1].axvline(vis_df_tex['Texture'].mean(), color='#6B4C8E',
               linestyle='--', lw=2,
               label=f"Mean = {vis_df_tex['Texture'].mean():.2f}")
axes[1].set_xlabel('Texture (Intra-block Std Dev)', fontsize=10)
axes[1].set_title('H\u00ecnh 6b \u2014 Texture by Category',
                  fontsize=12, fontweight='bold')
axes[1].set_facecolor(BG_NB)
axes[1].tick_params(axis='y', labelsize=8)
axes[1].set_xlim(0.05, 0.21)
for s in axes[1].spines.values():
    s.set_visible(False)
axes[1].xaxis.set_visible(False)
axes[1].legend(fontsize=9, loc='lower right')

plt.suptitle('H\u00ecnh 6 \u2014 Image Visual Analysis: Brightness & Texture',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(REPORT_FIGS / 'ia_02_image_visual_analysis.png', dpi=180, bbox_inches='tight')
plt.show()

# ── Hình 7: RGB Channel Dominance by Thematic Cluster ───────────────────
fig2, ax2 = plt.subplots(figsize=(10, 5))
fig2.patch.set_facecolor(BG_NB)

CHANNEL_COLORS = {'G>R>B': '#4CAF50', 'B>R>G': '#2196F3',
                  'R≈G>B': '#FF9800', 'R>G>B': '#795548'}
ch_counts = vis_df['Dom. Channel'].value_counts()
ch_labels = ch_counts.index.tolist()
ch_colors = [CHANNEL_COLORS.get(c, '#9E9E9E') for c in ch_labels]

bars2 = ax2.bar(ch_labels, ch_counts.values, color=ch_colors,
               edgecolor='none', width=0.55)
for bar, val in zip(bars2, ch_counts.values):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
             str(val), ha='center', fontsize=11, fontweight='bold')

ax2.set_xlabel('Dominant RGB Channel Pattern', fontsize=10)
ax2.set_ylabel('Number of Categories', fontsize=10)
ax2.set_title('H\u00ecnh 7 \u2014 RGB Channel Dominance Across 33 Categories',
              fontsize=12, fontweight='bold')
ax2.set_facecolor(BG_NB)
ax2.tick_params(axis='x', labelsize=10)
for s in ax2.spines.values():
    s.set_visible(False)
ax2.yaxis.set_visible(False)
ax2.grid(axis='y', color='#D4C9B8', linewidth=0.5, zorder=0)

# legend
legend_patches = [plt.Rectangle((0,0), 1, 1, facecolor=c)
                   for c in CHANNEL_COLORS.values()]
legend_labels2  = list(CHANNEL_COLORS.keys())
ax2.legend(legend_patches, legend_labels2,
           loc='upper right', frameon=False, fontsize=9)

plt.tight_layout()
plt.savefig(REPORT_FIGS / 'ia_03_rgb_channel_dominance.png', dpi=180, bbox_inches='tight')
plt.show()

print(f"\nSaved: ia_02_image_visual_analysis.png, ia_03_rgb_channel_dominance.png")


### 2.2 Caption Characteristics

In [9]:
# Collect caption-level stats
train_caps = [s['raw'] for img in train_imgs for s in img['sentences']]
test_caps  = [s['raw'] for img in test_imgs  for s in img['sentences']]

train_word_counts = [len(tokenize(cap)) for cap in train_caps]
test_word_counts  = [len(tokenize(cap)) for cap in test_caps]

train_raw_words   = [w for cap in train_caps for w in tokenize(cap)]
train_clean_words = [w for cap in train_caps for w in tokenize(cap, True)]
test_clean_words  = [w for cap in test_caps  for w in tokenize(cap, True)]

print(f"Train captions : {len(train_caps):,}")
print(f"Test  captions : {len(test_caps):,}")
print(f"Train avg words/caption: {np.mean(train_word_counts):.2f} ± {np.std(train_word_counts):.2f}")
print(f"Test  avg words/caption: {np.mean(test_word_counts):.2f} ± {np.std(test_word_counts):.2f}")
print(f"Min words: {min(train_word_counts)}, Max words: {max(train_word_counts)}")

# Vocabulary overlap
shared = len(set(train_clean_words) & set(test_clean_words))
total_test = len(set(test_clean_words))
overlap_pct = shared / total_test * 100
print(f"\nVocabulary (train, clean)  : {len(set(train_clean_words)):,}")
print(f"Vocabulary (test, clean)   : {len(set(test_clean_words)):,}")
print(f"Shared vocabulary          : {shared:,} ({overlap_pct:.1f}% of test)")

In [10]:
# Hình 2: Top-10 content words (train, stopwords removed)
word_freq = Counter(train_clean_words)
top10 = word_freq.most_common(10)

fig, ax = plt.subplots(figsize=(9, 5))
words, counts = zip(*top10)
palette = plt.cm.Blues(np.linspace(0.4, 0.9, len(words)))[::-1]
bars = ax.barh(list(words)[::-1], list(counts)[::-1], color=palette)
for bar, val in zip(bars, list(counts)[::-1]):
    ax.text(val + 30, bar.get_y()+bar.get_height()/2, f'{val:,}',
            va='center', fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Hình 2 — Top 10 Content Words (Training Set)', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(counts)*1.15)
plt.tight_layout()
plt.savefig(REPORT_FIGS / 'ta_02_word_frequency_train.png', dpi=180, bbox_inches='tight')
plt.show()

In [11]:
# Bigrams
all_bigrams = list(ngrams(train_clean_words, 2))
bigram_freq = Counter(all_bigrams).most_common(10)

fig, ax = plt.subplots(figsize=(9, 5))
labels = [' '.join(bg) for bg, _ in bigram_freq]
vals   = [c for _, c in bigram_freq]
palette = plt.cm.Oranges(np.linspace(0.4, 0.9, len(labels)))[::-1]
bars = ax.barh(labels[::-1], vals[::-1], color=palette)
for bar, val in zip(bars, vals[::-1]):
    ax.text(val + 20, bar.get_y()+bar.get_height()/2, f'{val:,}',
            va='center', fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Hình 3 — Top 10 Bigrams (Training Set)', fontsize=12, fontweight='bold')
ax.set_xlim(0, max(vals)*1.15)
plt.tight_layout()
plt.savefig(REPORT_FIGS / 'ta_04_bigram_frequency_train.png', dpi=180, bbox_inches='tight')
plt.show()

### 2.3 Visual Language in Captions

In [12]:
COLOR_WORDS = {'green','white','red','blue','gray','grey','yellow','dark',
               'black','brown','light','orange','bright','purple','colored','pink',
               'silver','golden'}

# Per-caption color presence
caps_with_color = sum(1 for cap in train_caps
                     if any(w in tokenize(cap) for w in COLOR_WORDS))

# Per-image color presence (>=1 caption mentions color)
imgs_with_color = sum(1 for img in train_imgs
                     if any(w in tokenize(s['raw'], False) for s in img['sentences']
                             for w in COLOR_WORDS))

color_pct_caps  = caps_with_color / len(train_caps) * 100
color_pct_imgs  = imgs_with_color / len(train_imgs) * 100

print(f"Captions mentioning color : {caps_with_color:,} / {len(train_caps):,} ({color_pct_caps:.1f}%)")
print(f"Images with color mention : {imgs_with_color:,} / {len(train_imgs):,} ({color_pct_imgs:.1f}%)")

# Color word frequency
color_freq = {w: c for w, c in Counter(train_raw_words).items() if w in COLOR_WORDS}
total_color = sum(color_freq.values())
color_df = pd.DataFrame(
    sorted(color_freq.items(), key=lambda x: -x[1]),
    columns=['Color','Count']
)
color_df['% of All Caps'] = (color_df['Count'] / len(train_caps) * 100).round(2)
color_df['% of Color Words'] = (color_df['Count'] / total_color * 100).round(2)

print("\nTable 2 — Color Word Frequency (Training Set)")
print(color_df.head(10).to_string(index=False))

In [13]:
# Hình color distribution
fig, ax = plt.subplots(figsize=(10, 5))
top_colors = color_df.head(10)
palette = plt.cm.Greens(np.linspace(0.35, 0.85, len(top_colors)))[::-1]
bars = ax.barh(top_colors['Color'].tolist()[::-1],
               top_colors['Count'].tolist()[::-1],
               color=palette)
for bar, val in zip(bars, top_colors['Count'].tolist()[::-1]):
    ax.text(val + 50, bar.get_y()+bar.get_height()/2, f'{val:,}', va='center', fontsize=9)
ax.set_xlabel('Frequency')
ax.set_title('Color Word Frequency — Training Captions', fontsize=12, fontweight='bold')
ax.set_xlim(0, top_colors['Count'].max()*1.15)
plt.tight_layout()
plt.savefig(REPORT_FIGS / 'mm_02_color_distribution_train.png', dpi=180, bbox_inches='tight')
plt.show()

In [14]:
# Spatial relationships
SPATIAL_PREPS = {
    'near': 0, 'surrounded': 0, 'next to': 0, 'around': 0, 'beside': 0,
    'in the middle of': 0, 'in the center of': 0, 'above': 0, 'below': 0
}
for cap in train_caps:
    cl = cap.lower()
    for prep in SPATIAL_PREPS:
        if prep in cl:
            SPATIAL_PREPS[prep] += 1

caps_with_spatial = sum(1 for cap in train_caps
                        if any(p in cap.lower() for p in SPATIAL_PREPS))
imgs_with_spatial = sum(1 for img in train_imgs
                        if any(p in s['raw'].lower()
                               for s in img['sentences'] for p in SPATIAL_PREPS))

print(f"Captions with spatial preposition : {caps_with_spatial:,} ({caps_with_spatial/len(train_caps)*100:.1f}%)")
print(f"Images with spatial mention       : {imgs_with_spatial:,} ({imgs_with_spatial/len(train_imgs)*100:.1f}%)")

print("\nTable 3 — Spatial Preposition Frequency (Training Set)")
spatial_df = pd.DataFrame([
    {'Preposition': k, 'Count': v, '% Caps': round(v/len(train_caps)*100,1)}
    for k, v in sorted(SPATIAL_PREPS.items(), key=lambda x: -x[1])
    if v > 0
])
print(spatial_df.to_string(index=False))

### 2.4 Caption Variability and Sample Pairs

In [15]:
# Variability: std of word counts across the 5 captions per image
variabilities = []
caption_data = []
for img in train_imgs:
    lens = [len(s['tokens']) for s in img['sentences']]
    std  = np.std(lens) if len(lens)>1 else 0
    variabilities.append(std)
    caption_data.append({
        'filename' : img['filename'],
        'category' : parse_filename(img['filename']),
        'mean_len' : np.mean(lens),
        'std_len'  : std,
        'captions' : [s['raw'] for s in img['sentences']],
    })

print(f"Mean caption-length std within image : {np.mean(variabilities):.2f} words")
print(f"Max  caption-length std within image : {np.max(variabilities):.2f} words")

In [16]:
# Hình 4 — Caption Variability
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: histogram of std
axes[0].hist(variabilities, bins=30, edgecolor='white', color='#9b59b6', alpha=0.85)
axes[0].axvline(np.mean(variabilities), color='red', linestyle='--', lw=2,
                label=f'Mean = {np.mean(variabilities):.2f}')
axes[0].set_xlabel('Caption-Length Std (within same image)')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Within-Image Caption Length Variability')
axes[0].legend()

# Right: mean std by category (top 10)
cat_var = defaultdict(list)
for item in caption_data:
    cat_var[item['category']].append(item['std_len'])
cat_var_sorted = sorted([(c, np.mean(stds)) for c, stds in cat_var.items()],
                        key=lambda x: x[1], reverse=True)[:10]

cats_, means_ = zip(*cat_var_sorted)
axes[1].barh(list(cats_)[::-1], list(means_)[::-1], color='#3498db')
axes[1].set_xlabel('Mean Caption-Length Std')
axes[1].set_title('Caption Variability by Category (Top 10)')

plt.suptitle('Hình 4 — Caption Variability Analysis (Training Set)',
             fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(REPORT_FIGS / 'mm_01_caption_variability_train.png', dpi=180, bbox_inches='tight')
plt.show()

In [17]:
# Hình 5 — Sample Image-Caption Pairs (text-only)
SENTENCE_COLORS = {0: '#e74c3c', 1: '#3498db', 2: '#2ecc71', 3: '#f39c12', 4: '#9b59b6'}

fig, axes = plt.subplots(3, 1, figsize=(14, 14))

for i, (ax, sample) in enumerate(zip(axes, caption_data[:3])):
    # Image label header
    ax.text(0.5, 0.97, f"Image: {sample['filename']}  |  Category: {sample['category']}",
            transform=ax.transAxes, fontsize=11, fontweight='bold',
            ha='center', va='top',
            bbox=dict(boxstyle='round,pad=0.4', facecolor='wheat', alpha=0.6))

    # Show up to 5 captions
    for j, cap in enumerate(sample['captions']):
        y_pos = 0.78 - j*0.19
        color = SENTENCE_COLORS[j]
        ax.text(0.03, y_pos,
                f"[{j+1}] {cap}",
                transform=ax.transAxes,
                fontsize=9.5,
                color=color,
                wrap=True,
                verticalalignment='top',
                bbox=dict(boxstyle='round,pad=0.3', facecolor=color, alpha=0.07))

    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis('off')
    ax.set_title(f'Sample {i+1}', fontsize=10, color='gray')

plt.suptitle('Hình 5 — Sample Image-Caption Pairs (Training Set)',
             fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.savefig(REPORT_FIGS / 'mm_03_sample_pairs_train.png', dpi=180, bbox_inches='tight')
plt.show()

---
## 3. Key Findings

In [18]:
# Summary statistics for key findings
top_obj_freq = Counter(train_clean_words).most_common(10)

print("=" * 60)
print("KEY FINDINGS — RSITMD Multimodal EDA")
print("=" * 60)
print("""
1. Visual-Linguistic Alignment is Strong but Templated
   - Color words in captions: {:.1f}% (captions), {:.1f}% (images)
   - Spatial prepositions in captions: {:.1f}% (captions), {:.1f}% (images)
   - Captions average {:.1f} words with std={:.2f} — tight template.

2. Severe Class Imbalance (37.8:1)
   - Most: storagetanks (227), Least: plane (6)
   - Majority categories dominate vocabulary statistics.

3. Compact Vocabulary ({:,} clean words)
   - Fixed-length text encoding is feasible.
   - Low OOV risk: {:.1f}% test vocabulary overlaps train.

4. Color is the Dominant Visual Feature
   - 'green' = {:.1f}% of all color word occurrences.

5. Spatial Relationships are Consistently Annotated
   - Most common: 'near' ({:,}), 'surrounded' ({:,})
""".format(
    color_pct_caps, color_pct_imgs,
    caps_with_spatial/len(train_caps)*100, imgs_with_spatial/len(train_imgs)*100,
    np.mean(train_word_counts), np.std(train_word_counts),
    len(set(train_clean_words)),
    overlap_pct,
    color_freq.get('green', 0),
    color_freq.get('green',0)/
        sum(color_freq.values())*100,
    SPATIAL_PREPS['near'], SPATIAL_PREPS['surrounded']

6. Image Visual Statistics (Pixel-Level)
   - Brightness: farmland=0.64 (brightest) to industrial=0.44 (darkest)
   - Texture: forest=0.18 (highest) to bareland=0.07 (lowest)
   - RGB dominance: G>R>B in vegetation, B>R>G in coastal, R≈G>B in urban)

In [19]:
# Final summary table — now includes image pixel-level metrics
summary = {
    'Metric'         : ['Train Images','Test Images','Train Captions','Test Captions',
                        'Captions/Image','Categories','Vocab (clean)','Avg Words/Caption',
                        'Color % Caps','Spatial % Caps','Imbalance Ratio',
                        'Brightness Range','Texture Range','RGB Patterns'],
    'Value'          : [4_291, 452, 21_455, 2_260, 5, 33, 3_470, 10.26,
                        '41.5%', '31.3%', '37.8:1',
                        '0.44–0.71', '0.07–0.18', '4 distinct patterns'],
    'Source'         : ['Report §1.1','Report §1.1','Report §1.1','Report §1.1',
                        'Report §1.1','Report §1.2','Report §2.2','Report §2.2',
                        'Report §2.3','Report §2.3','Report §2.1',
                        'Report §2.2','Report §2.2','Report §2.2'],
}

summary_df = pd.DataFrame(summary)
summary_df.index = range(1, len(summary_df)+1)
summary_df.index.name = '#'
print("\nSummary Table — RSITMD Multimodal EDA")
print(summary_df.to_string())

In [20]:
# Save all figures
print("All figures saved to:", REPORT_FIGS)
for f in sorted(REPORT_FIGS.glob("*.png")):
    print(f"  {f.name}")